# Phase 4 : Model Training

This notebook trains and evaluates multiple machine learning models for customer churn prediction.

Objectives

- Load processed datasets
- Train multiple classification models
- Compare model performance
- Tune the best-performing model
- Save the final model

# Import Libraries

In [21]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

# Load Processed Dataset

In [22]:
X_train = pd.read_csv("../data/processed/X_train.csv")

X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [23]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(5634, 51)
(1409, 51)
(5634,)
(1409,)


# Load Preprocessor

In [24]:
preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

In [25]:
# Verify
type(preprocessor)

sklearn.compose._column_transformer.ColumnTransformer

# Model Evaluation Function

In [26]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    results = {

        "Accuracy": accuracy_score(y_test, predictions),

        "Precision": precision_score(y_test, predictions),

        "Recall": recall_score(y_test, predictions),

        "F1 Score": f1_score(y_test, predictions),

        "ROC AUC": roc_auc_score(y_test, probabilities)

    }

    return results

# Initialize Models

In [27]:
models = {

    "Logistic Regression":

        LogisticRegression(
            random_state=42,
            max_iter=1000
        ),

    "Decision Tree":

        DecisionTreeClassifier(
            random_state=42
        ),

    "Random Forest":

        RandomForestClassifier(
            random_state=42
        ),

    "Gradient Boosting":

        GradientBoostingClassifier(
            random_state=42
        )

}

# Train Models

In [28]:
results = []

for name, model in models.items():

    metrics = evaluate_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    metrics["Model"] = name

    results.append(metrics)

# Compare Models

In [29]:
results_df = pd.DataFrame(results)

results_df = results_df[

    [

        "Model",

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score",

        "ROC AUC"

    ]

]

results_df = results_df.sort_values(

    by="ROC AUC",

    ascending=False

)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
3,Gradient Boosting,0.799858,0.652318,0.526738,0.582840,0.852257
0,Logistic Regression,0.801278,0.647799,0.550802,0.595376,0.849319
2,Random Forest,0.791341,0.634228,0.505348,0.562500,0.833411
1,Decision Tree,0.737402,0.505291,0.510695,0.507979,0.664773


In [30]:
# Save Result
results_df.to_csv(

    "../reports/model_comparison.csv",

    index=False

)

# Select Best Model

In [31]:
best_model_name = results_df.iloc[0]["Model"]

best_model_name

'Gradient Boosting'

In [32]:
best_model = models[best_model_name]

best_model.fit(

    X_train,

    y_train

)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

# Confusion Matrix

In [33]:
predictions = best_model.predict(X_test)

cm = confusion_matrix(

    y_test,

    predictions

)

cm

array([[930, 105],
       [177, 197]])

In [34]:
# Classification Report
print(

    classification_report(

        y_test,

        predictions

    )

)

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.53      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409



# Cross Validation

In [35]:
cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

In [36]:
scores = cross_val_score(

    best_model,

    X_train,

    y_train,

    cv=cv,

    scoring="roc_auc"

)

scores

array([0.85739704, 0.84595996, 0.85980442, 0.8713344 , 0.87550602])

In [37]:
print(

    scores.mean()

)

0.86200036861994


# Save Best Model

In [38]:
joblib.dump(

    best_model,

    "../models/churn_prediction_model.pkl"

)

['../models/churn_prediction_model.pkl']

In [39]:
# Verify
import os

os.path.exists(

    "../models/churn_prediction_model.pkl"

)

True

# Training Summary

In [40]:
summary = {

    "Training Samples":

        X_train.shape[0],

    "Testing Samples":

        X_test.shape[0],

    "Features":

        X_train.shape[1],

    "Best Model":

        best_model_name,

    "Best ROC AUC":

        round(

            results_df.iloc[0]["ROC AUC"],

            4

        )

}

pd.DataFrame(

    summary.items(),

    columns=[

        "Metric",

        "Value"

    ]

)

,Metric,Value
0,Training Samples,5634
1,Testing Samples,1409
2,Features,51
3,Best Model,Gradient Boosting
4,Best ROC AUC,0.8523
